In [27]:
import pulp
from pulp import LpProblem, LpMaximize
import numpy as np

In [28]:
# init model
model = LpProblem("TeaDistributionTask", LpMaximize)

In [29]:
# declare variables
tea_types = ['GT', 'BT', 'WT', 'RT']
box_sizes = [50, 100]
countries = ['PT', 'ES', 'FR', 'IT', 'DE', 'PL']

x = pulp.LpVariable.dicts(
    "x",
    ((i, j, k) for i in tea_types for j in box_sizes for k in countries),
    lowBound=0,
    cat='Integer'
)

In [30]:
# selling prices
P = np.array([
    [6, 10],
    [7, 12],
    [9, 16],
    [8, 14]
])

# raw tea cost # additional constraint may be needed (if we sell odd number of 50g boxes of one kind, we must add the full price because we can only buy 100g boxes)
C = np.array([
    [1.75, 3.5],
    [2.25, 4.5],
    [2.75, 5.5],
    [2.5, 5.0]
])

# distribution cost per country
D = np.array([0.5, 0.4, 0.6, 0.7, 0.8, 1.0])

In [31]:
# index mappings
i_index = {tea_types[i]: i for i in range(len(tea_types))}
j_index = {box_sizes[j]: j for j in range(len(box_sizes))}
k_index = {countries[k]: k for k in range(len(countries))}

In [32]:
# objective function

model += pulp.lpSum(
    (P[i_index[i], j_index[j]] - C[i_index[i], j_index[j]] - D[k_index[k]] - 1) * x[i, j, k]
    for i in tea_types
    for j in box_sizes
    for k in countries
) - 400000 - 4000000

In [33]:
# constraints

# max total number of boxes produced
model += pulp.lpSum(
    x[i, j, k]
    for i in tea_types
    for j in box_sizes
    for k in countries
) <= 2 * 1e6

# max amount of available tea
model += pulp.lpSum(j * x['GT', j, k] for j in box_sizes for k in countries) <= 25 * 1e6
model += pulp.lpSum(j * x['BT', j, k] for j in box_sizes for k in countries) <= 30 * 1e6
model += pulp.lpSum(j * x['WT', j, k] for j in box_sizes for k in countries) <= 15 * 1e6
model += pulp.lpSum(j * x['RT', j, k] for j in box_sizes for k in countries) <= 20 * 1e6

# min 10% of total production
total = pulp.lpSum(x[i, j, k] for i in tea_types for j in box_sizes for k in countries)
model += pulp.lpSum(x['GT', j, k] for j in box_sizes for k in countries) >= total * 0.1
model += pulp.lpSum(x['BT', j, k] for j in box_sizes for k in countries) >= total * 0.1
model += pulp.lpSum(x['WT', j, k] for j in box_sizes for k in countries) >= total * 0.1
model += pulp.lpSum(x['RT', j, k] for j in box_sizes for k in countries) >= total * 0.1

# demand forecast
F = np.array([
    # GT (Green Tea)
    [
        [100, 150, 200, 130, 180, 120],  # 50g
        [ 80, 120, 170, 110, 160, 100]   # 100g
    ],
    # BT (Black Tea)
    [
        [ 90, 130, 180, 120, 170, 110],  # 50g
        [ 70, 110, 160, 100, 150,  90]   # 100g
    ],
    # WT (White Tea)
    [
        [ 50,  80, 100,  70,  90,  60],  # 50g
        [ 40,  60,  90,  60,  80,  50]   # 100g
    ],
    # RT (Red Tea)
    [
        [ 60, 100, 120,  90, 110,  80],  # 50g
        [ 50,  90, 110,  80, 100,  70]   # 100g
    ]
])

for i in tea_types:
    for j in box_sizes:
        for k in countries:
            model += x[i, j, k] <= F[i_index[i], j_index[j], k_index[k]]

In [34]:
# solving the problem
model.solve()

1

## Results

In [35]:
total_boxes = sum(x[i, j, k].varValue 
                  for i in tea_types 
                  for j in box_sizes 
                  for k in countries)

print("Total number of boxes:", total_boxes)

Total number of boxes: 4990.0


In [36]:
for k in countries:
    country_boxes = sum(x[i, j, k].varValue for i in tea_types for j in box_sizes)
    print(f"Total number of boxes in {k}:", country_boxes)

Total number of boxes in PT: 540.0
Total number of boxes in ES: 840.0
Total number of boxes in FR: 1130.0
Total number of boxes in IT: 760.0
Total number of boxes in DE: 1040.0
Total number of boxes in PL: 680.0


In [37]:
for i in tea_types:
    country_boxes = sum(x[i, j, k].varValue for j in box_sizes for k in countries)
    print(f"Total number of boxes of {i}:", country_boxes)

Total number of boxes of GT: 1620.0
Total number of boxes of BT: 1480.0
Total number of boxes of WT: 830.0
Total number of boxes of RT: 1060.0


In [38]:
for j in box_sizes:
    country_boxes = sum(x[i, j, k].varValue for i in tea_types for k in countries)
    print(f"Total number of boxes of size {j}g:", country_boxes)

Total number of boxes of size 50g: 2690.0
Total number of boxes of size 100g: 2300.0


In [39]:
revenue = sum(P[i_index[i], j_index[j]] * x[i, j, k].varValue for i in tea_types for j in box_sizes for k in countries)
print(f"Total revenue: {revenue}")

Total revenue: 48050.0


In [40]:
revenue = sum(P[i_index[i], j_index[j]] * x[i,j,k].varValue 
              for i in tea_types 
              for j in box_sizes 
              for k in countries)

raw_cost = sum(C[i_index[i], j_index[j]] * x[i,j,k].varValue 
               for i in tea_types 
               for j in box_sizes 
               for k in countries)

dist_cost = sum(D[k_index[k]] * x[i,j,k].varValue 
                for i in tea_types 
                for j in box_sizes 
                for k in countries)

prod_mark_cost = sum(1 * x[i,j,k].varValue
                     for i in tea_types 
                     for j in box_sizes 
                     for k in countries)

fixed_cost = 400000 + 4000000

operating_income = revenue - (raw_cost + dist_cost + prod_mark_cost + fixed_cost)

print("Revenue:", revenue)
print("Raw tea cost:", raw_cost)
print("Distribution cost:", dist_cost)
print("Production/Marketing cost:", prod_mark_cost)
print("Fixed cost:", fixed_cost)
print("Operating income:", operating_income)

Revenue: 48050.0
Raw tea cost: 16217.5
Distribution cost: 3328.0
Production/Marketing cost: 4990.0
Fixed cost: 4400000
Operating income: -4376485.5


In [41]:
# weight constraint
gram_constr = [25, 30, 15, 20]
for i in tea_types:
    tea_grams = sum(j * x[i, j, k].varValue for j in box_sizes for k in countries)
    print(f"Grams of {i}: {tea_grams}, within constraint: {tea_grams <= gram_constr[i_index[i]] * 1e6}")

Grams of GT: 118000.0, within constraint: True
Grams of BT: 108000.0, within constraint: True
Grams of WT: 60500.0, within constraint: True
Grams of RT: 78000.0, within constraint: True


In [42]:
# market presence constraint
total_boxes = sum(x[i, j, k].varValue for i in tea_types for j in box_sizes for k in countries)
for i in tea_types:
    tea_boxes = sum(x[i, j, k].varValue for j in box_sizes for k in countries)
    print(f"Within market presence constraint for {i}: {tea_boxes >= total_boxes * 0.1}")

Within market presence constraint for GT: True
Within market presence constraint for BT: True
Within market presence constraint for WT: True
Within market presence constraint for RT: True


In [43]:
# Identify which constraints (demand, capacity, raw tea availability, market presence) are violated in a given distribution
# plan. ???

In [44]:
print(pulp.LpStatus[model.status])

Optimal
